In [44]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import gzip

from collections import Counter
from datetime import datetime

from utils import loadmat

DATES = ["020425",
    "013125",
    "012925",
    "013025",
    "021125",
    "040925",
    "041125",
    "041625",
    "020725",
    "121725",
    "042825",
    "041825",
    "121925",
    "121825",
    "042425",
    "042525",
    "121125",
    "021825",
    "021925",
    "121625",
    "121025",
    "020525",
    "052926",
    "060326", 
    "070126",
]

RAW_ROOT = "../data/rawdata/"
NEURAL_DIR = os.path.join("../data", "procdata")

TASK_DIR = {
    "probe": os.path.join(RAW_ROOT, "stim_data", "probe"),
    "plaid": os.path.join(RAW_ROOT, "stim_data", "plaid"),
    "video": os.path.join(RAW_ROOT, "stim_data", "video"),
}

VIDEO_DIMS_PX = (640, 360)
PROBE_MONITOR_LATENCIES = np.array([0.03, 0.02, 0.02], dtype=float)
TRIAL_BUFFER=4.0  # all trials are less than 4 seconds
PRE_TRIAL_TIME = 0.3 # pad spike train with 300 ms before fixation acquisition
POST_TRIAL_TIME = 0.5 # pad spike train with 500 ms after trial end

NUM_CHAN = 16 # number of channels for saving multichannel waveform

TASK_TO_DURATION = {"probe": 100, "plaid": 250, "video": 200}

PROBE_TIME_BEFORE = 100 # we include 100 ms before each probe onset in the probe-aligned dataset for ease of use and visualization 
PROBE_TIME_AFTER = 100 # we include 100 ms after each probe onset in the probe-aligned dataset for ease of use and visualization
 
PDIODE_ONLY_DATE = "060326"   # the sole session with photodiode but no sync

In [39]:
def get_ptb_path(date, exp, task):
    return os.path.join(RAW_ROOT, f"stim_data/{task}/tg_20{date[-2:] + date[0:-2]}_{exp}_data_structure.mat")

def find_exp(date, task):
    exps = []
    for exp in range(20):
        ptb_path = get_ptb_path(date, exp, task)
        if os.path.exists(ptb_path):
            exps.append(exp)            
    return exps

def compare_date(input_date: str) -> bool:
    # input_date is in MMDDYY, returns True if input_date > 2023-08-04
    month = int(input_date[0:2])
    day   = int(input_date[2:4])
    year  = 2000 + int(input_date[4:6])

    input_dt = datetime(year, month, day)
    reference_dt = datetime(2023, 8, 4)

    return input_dt > reference_dt

def load_neural_npz(npz_path):
    z = np.load(npz_path, allow_pickle=True)
    probe_idx = 0
    neural_data = {
        probe_idx: {
            "times": z["times"],
            "ids": z["ids"],
            "x_coords": z["x_coords"],
            "y_coords": z["y_coords"],
            "waveforms": z["waveforms"],
        },
        "sync_on": z["sync_on"],
        "sync_off": z["sync_off"],
        "srate": float(z["srate"]),
    }
    neural_data["has_pdiode"] = bool(z["has_pdiode"]) if "has_pdiode" in z.files else False
    if "pdiode_on" in z.files:
        neural_data["pdiode_on"] = z["pdiode_on"]
        neural_data["pdiode_off"] = z["pdiode_off"]
    else:
        neural_data["pdiode_on"] = np.array([])
        neural_data["pdiode_off"] = np.array([])

    if not neural_data["has_pdiode"]:
        neural_data["pdiode_on"] = np.array([])
        neural_data["pdiode_off"] = np.array([])

    return neural_data, probe_idx

def build_sync_off_from_pdiode(neural_data, ptb_data):
    pon = np.asarray(neural_data["pdiode_on"], dtype=float)
    stim = ptb_data["expsetup"]["stim"]
 
    probe_on_raw = stim["edata_probe_on"]
    first_on = np.array([
        np.nanmin(np.atleast_1d(np.asarray(r, dtype=float)).ravel())
        if np.any(np.isfinite(np.atleast_1d(np.asarray(r, dtype=float)))) else np.nan
        for r in np.atleast_1d(probe_on_raw)
    ])
    loop_over = np.asarray(stim["edata_loop_over"], dtype=float)
 
    has_stim = np.isfinite(first_on)
    stim_idx = np.flatnonzero(has_stim)
 
    n_cov = len(pon)
    cov = stim_idx[:n_cov] 
    surr_measured = pon + (loop_over[cov] - first_on[cov])
 
    A = np.polyfit(loop_over[cov], surr_measured, 1)

    last = int(cov[-1])
    sync_off = np.polyval(A, loop_over).astype(float)
    sync_off[cov] = surr_measured               
    if last + 1 < len(sync_off):
        n_tail = len(sync_off) - (last + 1)
        big = sync_off[last] + 1e6 + np.arange(1, n_tail + 1) * 1e3
        sync_off[last + 1:] = big
 
    return sync_off

def load_and_sync(neural_path, ptb_path, date):
    neural_data, probe_idx = load_neural_npz(neural_path)
    
    print(f"keys: {neural_data.keys()}")
    print(f"probe 0 keys: {neural_data[probe_idx].keys()}")
    print(f"number of single units: {len(np.unique(neural_data[probe_idx]['ids']))}")
    print(f"number of sync signals: {len((neural_data['sync_on']))}")

    ptb_data = loadmat(ptb_path)
    print(f"keys: {ptb_data.keys()}")
    print(f"expsetup keys: {ptb_data['expsetup'].keys()}")
    print(f"number of sync signals: {len(ptb_data['expsetup']['stim']['edata_first_display'])}")
    
    if date == PDIODE_ONLY_DATE and len(neural_data["sync_off"]) == 0:
        neural_data["sync_off"] = build_sync_off_from_pdiode(neural_data, ptb_data)
        neural_data["sync_on"] = neural_data["sync_off"].copy()  # keep prints happy
    
    neu_sync = neural_data['sync_off']

    if compare_date(date):
        # in later versions of the task code, entire trial signal was sent for sync
        # thus, we aligned ptb_sync "loop_over" to "sync_off"
        ptb_sync = ptb_data['expsetup']['stim']['edata_loop_over']
        neu_sync_a = neu_sync - neu_sync[0] # times relative to first sync
        ptb_sync_a = ptb_sync - ptb_sync[0] # times relative to first sync
    else:
        # in early versions of the task code, only start trial was sent for sync
        # thus, we aligned ptb_sync "edata_first_display" to "sync_off"
        ptb_sync = ptb_data['expsetup']['stim']['edata_first_display']
        
        # we skip the first index because this has a messy WaitSecs call that messes up the sync on trial 1
        # this means that later once we find the 'match idx', we need to subtract 1
        ptb_sync = ptb_sync[1:] 

        neu_sync_a = neu_sync - neu_sync[0] # times relative to first sync
        ptb_sync_a = ptb_sync - ptb_sync[0] # times relative to first sync
    
    # when syncing we need to handle these cases: 
    # 1) the recording ended before the task ended, so we need to remove ptb_sync signals until we're at the end of neu_sync
    # 2) the neu_sync_signal is longer than the ptb_sync_signal, so we need to align somewhere to the middle of the neu recording
    done_sync = False
    while not done_sync and len(ptb_sync_a) > 10:    
        dist = np.zeros(len(neu_sync_a) - len(ptb_sync_a) + 1) # array containing error at different alignment positions
        for i in range(len(neu_sync_a)):
            # edge case 2
            if i + len(ptb_sync_a) <= len(neu_sync_a):
                # rezero neu_sync based on current alignement guess, and aggregate ptb_syncs
                dist[i] = np.sqrt(np.sum(np.square(neu_sync_a[i:i+len(ptb_sync_a)] - neu_sync_a[i] - ptb_sync_a)))
        
        if min(dist) < 1:
            done_sync = True
        else:
            ptb_sync_a = ptb_sync_a[:-1] # edge case 1
            
    match_idx = np.argmin(dist) # distance is minimize when sync is correct 
    
    # deprecated plot command if we want to visualize the sync over trials
    #plt.plot(np.square(neu_sync_a[match_idx:match_idx+len(ptb_sync_a)] - neu_sync_a[match_idx] - ptb_sync_a))
    #plt.show()
    
    # see earlier comment
    if compare_date(date):
        pass
    else:
        match_idx = match_idx - 1
    
    num_trials = len(ptb_sync_a)
    num_neurons = len(np.unique(neural_data[probe_idx]['ids']))
    print(f"neural data is aligned to the {match_idx + 1}st ptb sync signal")

    # assert sync worked 
    assert min(dist) < 1
    
    return neural_data, ptb_data, match_idx, num_trials, num_neurons, probe_idx

def get_num_probes(ptb_data, num_trials):
    # returns max number of probe onsets per trial
    num_probes = 0
    for trial_idx in range(num_trials):
        probe_on = ptb_data['expsetup']['stim']['edata_probe_on'][trial_idx]
        if type(probe_on) == float:
            num_probes = max(num_probes, 1)
            continue
        num_probes = max(num_probes, len(probe_on))
    return num_probes

def get_ksort_info(date, probe_idx):
    kilosort_path = f"../data/rawdata/neural_data/{date}_kilosort4/kilosort4/"

    # location of each neuron and waveform info
    templates = np.load(kilosort_path + "templates.npy") # num_units x num_times x num_channels
    chan_map = np.load(kilosort_path + "channel_map.npy")
    clu = np.load(kilosort_path + 'spike_clusters.npy')
    ops = np.load(os.path.join(kilosort_path, "ops.npy"), allow_pickle=True).item()
    probe = ops['probe']
    timepoints = (np.arange(ops['nt']) / ops['fs']) * 1000
    assert probe['xc'].shape[0] == templates.shape[2]
        
    chan_best = (templates**2).sum(axis=1).argmax(axis=-1) # we take the channel w the max fluctuation to be the units's position

    x_coord, y_coord = probe['xc'][chan_best], probe['yc'][chan_best]
    
    waveforms = templates[np.arange(templates.shape[0]), :, chan_best]
    times = np.tile(timepoints, (waveforms.shape[0], 1))
    
    # get cluster amplitude and contam_pct
    camps = pd.read_csv(os.path.join(kilosort_path, 'cluster_Amplitude.tsv'), sep='\t')['Amplitude'].values
    contam_pct = pd.read_csv(os.path.join(kilosort_path, 'cluster_ContamPct.tsv'), sep='\t')['ContamPct'].values
    
    # get multichannel waveform info
    xc, yc = probe['xc'], probe['yc']
    
    mchan_xs, mchan_ys, mchan_waveforms = [], [], []
    for wi in range(camps.shape[0]):
        wv = templates[wi].copy()
        cb = chan_best[wi]
        nsp = (clu==wi).sum()

        n_chan = wv.shape[-1]
        ic0 = max(0, cb-NUM_CHAN//2)
        ic1 = min(n_chan, cb+NUM_CHAN//2)
        wv = wv[:, ic0:ic1]
        x0, y0 = xc[ic0:ic1], yc[ic0:ic1]
        
        mchan_xs.append(x0)
        mchan_ys.append(y0)
        mchan_waveforms.append(wv)

    return y_coord, x_coord, chan_best, waveforms, times, camps, contam_pct, mchan_xs, mchan_ys, mchan_waveforms

def get_neural_aligned(neural_data, ptb_data, num_trials, num_neurons, match_idx, date, num_probes, probe_idx):
    # assemble spike train and event times
    num_times = int((PRE_TRIAL_TIME + POST_TRIAL_TIME + TRIAL_BUFFER) * 1000)
    spike_train = np.zeros((num_trials, num_times, num_neurons), dtype=bool) # bool for memory
    times = np.linspace(-PRE_TRIAL_TIME, TRIAL_BUFFER + POST_TRIAL_TIME, num_times) # times relative to fixation acquisition 

    fixation_onset = np.full((num_trials, 1), np.nan, dtype=float)
    probe_onsets = np.full((num_trials, num_probes), np.nan, dtype=float)
    probe_offsets = np.full((num_trials, num_probes), np.nan, dtype=float)
    for trial_idx in range(num_trials):
        if compare_date(date):
            # sync off signal for the trial
            neural_trial_end = neural_data['sync_off'][trial_idx + match_idx]
        else:
            # sync_off signal + trial_time (end-start)
            neural_trial_end = neural_data['sync_off'][trial_idx + match_idx] + ptb_data['expsetup']['stim']['edata_loop_over'][trial_idx] - ptb_data['expsetup']['stim']['edata_first_display'][trial_idx]
            
        ptb_trial_end = ptb_data['expsetup']['stim']['edata_loop_over'][trial_idx]
        fix_onset = ptb_data['expsetup']['stim']['edata_fixation_acquired'][trial_idx]
        probe_onset = ptb_data['expsetup']['stim']['edata_probe_on'][trial_idx]
        probe_offset = ptb_data['expsetup']['stim']['edata_probe_off'][trial_idx]
        fix_dist_from_end = ptb_trial_end - fix_onset

        trial_start = neural_trial_end - fix_dist_from_end - PRE_TRIAL_TIME
        trial_end = neural_trial_end - fix_dist_from_end + TRIAL_BUFFER + POST_TRIAL_TIME

        include_idx = np.logical_and(neural_data[probe_idx]["times"] > trial_start, neural_data[probe_idx]["times"] < trial_end)

        trial_times = neural_data[probe_idx]["times"][include_idx] - trial_start
        trial_ids = neural_data[probe_idx]["ids"][include_idx] 

        # note there's a bit of 'weirdness' here with how we do the binning given that we corrected for MONITOR_LATENCY
        # this doesn't influence downstream data because we later ignore everything in spike_train 'late/after' the trial
        for neuron_idx in range(num_neurons):
            hist, bin_edges = np.histogram(trial_times[trial_ids == neuron_idx] * 1000, bins=np.arange(num_times + 1))
            spike_train[trial_idx, :, neuron_idx] += hist > 0

        fixation_onset[trial_idx] = 0
        
        # handle cases with 1 or more probes, can happen, e.g., with incorrect trials 
        if type(probe_onset) == float:
            probe_onset = [probe_onset]
        if type(probe_offset) == float:
            probe_offset = [probe_offset]

        probe_onset = np.array(probe_onset)
        probe_offset = np.array(probe_offset)
        
        # if had fewer probes than expected (e.g., incorrect trial), pad with nans to expected shape
        if len(probe_onset) < num_probes:
            probe_onset = np.pad(probe_onset, (0, num_probes - len(probe_onset)), constant_values=np.nan)
        if len(probe_offset) < num_probes:
            probe_offset = np.pad(probe_offset, (0, num_probes - len(probe_offset)), constant_values=np.nan)
        
        # record probe onset and offset (note we subtract fix_onset here such that it is probe onset and offset relative to fix onset)
        probe_onsets_trial = probe_onset - fix_onset
        probe_offsets_trial = probe_offset - fix_onset

        n_present = np.sum(~np.isnan(probe_onsets_trial))
        latencies = np.full(num_probes, np.nan, dtype=float)
        if n_present > 0:
            latencies[:n_present] = PROBE_MONITOR_LATENCIES[np.minimum(np.arange(n_present), len(PROBE_MONITOR_LATENCIES) - 1)]

        probe_onsets[trial_idx, :] = probe_onsets_trial + latencies
        probe_offsets[trial_idx, :] = probe_offsets_trial + latencies
        
    return spike_train, times, fixation_onset, probe_onsets, probe_offsets

def load_video_ids(date):
    path = os.path.join(TASK_DIR["video"], f"video_ids_{date}.mat")
    trials = loadmat(path)["image_ids"].squeeze().tolist()
    out = [np.array(trial.squeeze().tolist(), dtype=object) for trial in trials]

    return out

def _n_items(arr, extra_dim):
    a = np.asarray(arr, dtype=float)
    if extra_dim == 1:
        return 1 if a.ndim == 0 else a.shape[0]
    else:
        if a.ndim <= 1:
            return 1
        return a.shape[0]
 
 
def pad_ragged_array(ragged_array, extra_dim=1):
    ragged_array = list(np.atleast_1d(ragged_array))
    n_trials = len(ragged_array)
 
    max_len = 1
    for arr in ragged_array:
        max_len = max(max_len, _n_items(arr, extra_dim))
 
    if extra_dim == 1:
        padded = np.full((n_trials, max_len), np.nan, dtype=float)
        for i, arr in enumerate(ragged_array):
            a = np.atleast_1d(np.asarray(arr, dtype=float))
            padded[i, :a.shape[0]] = a
    else:
        padded = np.full((n_trials, max_len, extra_dim), np.nan, dtype=float)
        for i, arr in enumerate(ragged_array):
            a = np.asarray(arr, dtype=float)
            if a.ndim <= 1:
                a = a.reshape(1, -1)
            padded[i, :a.shape[0], :a.shape[1]] = a[:, :extra_dim]
 
    return padded

def get_probe_plaid_trial_info(num_probes, ptb_data):
    trial_correct = np.array(ptb_data['expsetup']['stim']['edata_trial_online_counter'])
 
    probe_id = pad_ragged_array(
        ptb_data['expsetup']['stim']['esetup_probe_grating_id'])[:, :num_probes]
    probe_coord = pad_ragged_array(
        ptb_data['expsetup']['stim']['esetup_probe_coord'], extra_dim=2)[:, :num_probes, :]
    probe_times = pad_ragged_array(
        ptb_data['expsetup']['stim']['edata_probe_on'])[:, :num_probes]
 
    # if probe was not 'on' need to mask other vars with nans
    probe_id[np.isnan(probe_times)] = np.nan
    probe_coord[:, :, 0][np.isnan(probe_times)] = np.nan
    probe_coord[:, :, 1][np.isnan(probe_times)] = np.nan
 
    return probe_id, probe_coord, trial_correct

def get_video_trial_info(num_probes, ptb_data, date):
    video_ids_by_trial = load_video_ids(date)

    trial_correct = np.array(ptb_data["expsetup"]["stim"]["edata_trial_online_counter"]).astype(bool)

    vid = np.array(video_ids_by_trial).squeeze()
    vid = vid[:, :num_probes]

    n_trials = vid.shape[0]
    probe_coord = np.full((n_trials, num_probes, 2), np.nan, dtype=float)

    counts = Counter(vid.reshape(-1))
    repeated = {k for k, v in counts.items() if v > 1}

    is_train_trial = np.zeros((n_trials,), dtype=bool)
    for t in range(n_trials):
        is_train_trial[t] = not (vid[t, 0] in repeated) # if the first stim was not in the test set, then it is a train trial

    return vid, probe_coord, trial_correct, is_train_trial, video_ids_by_trial

In [40]:
tasks = ["probe", "plaid", "video"]
for date in DATES:
    print(date)
    neuron_info = []
    session_info = []

    neural_path = os.path.join(NEURAL_DIR, f"{date}.npz")

    for task in tasks:  
        print(task)
        exps = find_exp(date, task)

        for exp in exps:
            print(exp)
            ptb_path = get_ptb_path(date, exp, task)
            neural_data, ptb_data, match_idx, num_trials, num_neurons, probe_idx = load_and_sync(neural_path, ptb_path, date)
            num_probes = get_num_probes(ptb_data, num_trials)
            y_coord, x_coord, chan_best, waveforms, waveform_times, camps, contam_pct, mchan_xs, mchan_ys, mchan_waveforms = get_ksort_info(date, probe_idx)
            spike_train, times, fixation_onset, probe_onsets, probe_offsets = get_neural_aligned(neural_data, ptb_data, num_trials, num_neurons, match_idx, date, num_probes, probe_idx)
            
            if task == "video":
                probe_id, probe_coord, trial_correct, is_train_trial, video_ids_by_trial = get_video_trial_info(num_probes, ptb_data, date)
            else:
                probe_id, probe_coord, trial_correct = get_probe_plaid_trial_info(num_probes, ptb_data)

            neuron_dict = {}
            neuron_dict['y'] = y_coord
            neuron_dict['x'] = x_coord
            neuron_dict['chan_best'] = chan_best
            neuron_dict['spike_train'] = spike_train
            neuron_dict['times'] = times
            neuron_dict['waveforms'] = waveforms
            neuron_dict['waveform_times'] = waveform_times
            neuron_dict['num_neurons'] = num_neurons
            neuron_dict['amps'] = camps
            neuron_dict['contam_pct'] = contam_pct
            neuron_dict['mchan_xs'] = mchan_xs
            neuron_dict['mchan_ys'] = mchan_ys
            neuron_dict['mchan_waveforms'] = mchan_waveforms

            session_dict = {}
            session_dict['fixation_onset'] = fixation_onset
            session_dict['probe_onsets'] = probe_onsets
            session_dict['probe_offsets'] = probe_offsets
            session_dict['probe_id'] = probe_id
            session_dict['probe_coord'] = probe_coord
            session_dict['trial_correct'] = trial_correct
            session_dict['num_probes'] = num_probes
            session_dict['probe_size'] = ptb_data['expsetup']['stim']['probe_size']
            
            session_dict['task'] = task
            session_dict['exps'] = exps
            session_dict['task_mod'] = ptb_data['expsetup']['stim']['exp_version_temp']

            if task == 'plaid':
                session_dict['plaid'] = ptb_data['expsetup']['stim']['gr_types_plaid']
                session_dict['direction'] = ptb_data['expsetup']['stim']['gr_types_ori']
            if task == 'probe':
                session_dict['direction'] = ptb_data['expsetup']['stim']['gr_types_ori']
            if task == "video":
                session_dict["is_train_trial"] = is_train_trial  # bool per trial
                session_dict["video_dims_px"] = VIDEO_DIMS_PX
                
                # in the video task, we sometimes offset the fixation location so we need to 
                # record some extra info to line things up later
                session_dict["deg2pix_screen"] = ptb_data["expsetup"]["screen"]["deg2pix"]
                session_dict["dispcenter_px"] = ptb_data["expsetup"]["screen"]["dispcenter"]
                session_dict["fix_coord_deg"] = ptb_data["expsetup"]["stim"]["esetup_fixation_coord"]

            neuron_info.append(neuron_dict)
            session_info.append(session_dict)
        
    with open(f"../data/procdata/output/output_{date}.pkl", "wb") as file:
        pickle.dump({'neuron': neuron_info, 'session': session_info}, file)

020425
probe
1
keys: dict_keys([0, 'sync_on', 'sync_off', 'srate', 'has_pdiode', 'pdiode_on', 'pdiode_off'])
probe 0 keys: dict_keys(['times', 'ids', 'x_coords', 'y_coords', 'waveforms'])
number of single units: 851
number of sync signals: 502
keys: dict_keys(['__header__', '__version__', '__globals__', 'expsetup', '__function_workspace__'])
expsetup keys: dict_keys(['general', 'screen', 'audio', 'ni_daq', 'tcpip', 'eyecalib', 'eyelink', 'stim'])
number of sync signals: 38
neural data is aligned to the 1st ptb sync signal
plaid
video
3
keys: dict_keys([0, 'sync_on', 'sync_off', 'srate', 'has_pdiode', 'pdiode_on', 'pdiode_off'])
probe 0 keys: dict_keys(['times', 'ids', 'x_coords', 'y_coords', 'waveforms'])
number of single units: 851
number of sync signals: 502
keys: dict_keys(['__header__', '__version__', '__globals__', 'expsetup', '__function_workspace__'])
expsetup keys: dict_keys(['general', 'screen', 'audio', 'ni_daq', 'tcpip', 'eyecalib', 'eyelink', 'stim'])
number of sync signals

In [41]:
def fetch_probe_spikes(spike_train, times, probe_onsets, probe_duration, trial_correct, probe_coords, probe_ids):
    n_trials = spike_train.shape[0]
    spikes = []
    coords = []
    ids = []
    trial_idx_out = []
    probe_idx_out = []
    
    times = np.around(times, 3) # round to nearest ms
    for trial_idx in range(n_trials):
        if trial_correct[trial_idx]:
            num_probes = np.sum(~np.isnan(probe_onsets[trial_idx]))
            for probe_idx in range(num_probes):
                start_time = np.around(probe_onsets[trial_idx, probe_idx].item(), 3) # round probe onset to nearest ms
                num_bins = probe_duration
                start_idx = np.searchsorted(times, start_time, side="left")
                end_idx = start_idx + num_bins
                spikes.append(spike_train[trial_idx, start_idx-PROBE_TIME_BEFORE:end_idx+PROBE_TIME_AFTER, :])
                coords.append(probe_coords[trial_idx, probe_idx, :])
                ids.append(probe_ids[trial_idx, probe_idx])
                trial_idx_out.append(trial_idx)
                probe_idx_out.append(probe_idx)

    spikes = np.stack(spikes)
    ids = np.stack(ids)
    coords = np.stack(coords)
    probe_times = np.array(np.arange(num_bins + PROBE_TIME_BEFORE + PROBE_TIME_AFTER) - PROBE_TIME_BEFORE)/1000.0
    trial_idx_out = np.array(trial_idx_out, dtype=int)
    probe_idx_out = np.array(probe_idx_out, dtype=int)
    return spikes, ids, coords, probe_times, trial_idx_out, probe_idx_out

In [ ]:
def get_stim_durations_ms(ptb_data, probe_onsets, probe_offsets, num_probes):
    dur = (probe_offsets - probe_onsets) * 1000.0  # (n_trials, num_probes)
    out = np.full(int(num_probes), np.nan)
    for j in range(int(num_probes)):
        col = dur[:, j]
        col = col[~np.isnan(col)]
        if not len(col):
            continue
        if len(col) >= 8:
            q1, q3 = np.percentile(col, [25, 75])
            trimmed = col[(col >= q1 - 1.5 * (q3 - q1)) & (col <= q3 + 1.5 * (q3 - q1))]
            if len(trimmed):
                col = trimmed
        out[j] = np.median(col)
        if np.std(col) > 5.0:
            print(f"      WARNING: stimulus {j} duration varies within session "
                  f"(median {out[j]:.1f} ms, std {np.std(col):.1f} ms) -- expected constant")

    if np.any(np.isnan(out)):
        declared = np.atleast_1d(
            np.asarray(ptb_data["expsetup"]["stim"]["probe_duration"], dtype=float)
        ).ravel()
        d = float(declared[0]) * 1000.0 if len(declared) else np.nan
        out[np.isnan(out)] = d

    return np.round(out).astype(int)

# loop over DATES and open the trial-aligned file we created abvoe
for date in DATES:
    print(date)
    with open(f"../data/procdata/output/output_{date}.pkl", "rb") as input_file:
        neural_data = pickle.load(input_file)
            
    # construct baseline_tasks, rf_tasks, plaid_tasks, and video_tasks which are lists of dicts 
    n_tasks = len(neural_data['session'])
    neuron_data = dict()
    
    rf_tasks = []
    plaid_tasks = []
    video_tasks = []
    baseline_tasks = []
    
    for i in range(n_tasks):
        task = neural_data["session"][i]["task"]

        spike_train = neural_data['neuron'][i]['spike_train']
        times = neural_data['neuron'][i]['times']
        probe_onsets = neural_data['session'][i]['probe_onsets']
        probe_offsets = neural_data['session'][i]['probe_offsets']
        probe_coords = neural_data['session'][i]['probe_coord']
        probe_ids = neural_data['session'][i]['probe_id']
        trial_correct = neural_data['session'][i]['trial_correct']
        num_probes = neural_data['session'][i]['num_probes']

        probe_duration = TASK_TO_DURATION[task] # in ms
        print(f"assumed stimulus duration: {probe_duration} ms")

        durations = get_stim_durations_ms(
            ptb_data, probe_onsets, probe_offsets, num_probes
        )
        probe_duration = int(np.nanmedian(durations)) # in ms
        print(f"derived stimulus duration: {probe_duration} ms")

        spikes, ids, coords, probe_times, trial_idx_out, probe_idx_out = fetch_probe_spikes(spike_train, times, probe_onsets, probe_duration, trial_correct, probe_coords, probe_ids)
        
        task_dict = {"spikes": spikes, "coords": coords, "times": probe_times, "task_index": i, "probe_duration": probe_duration}
        
        if task == "probe":
            sess = neural_data["session"][i]
            task_dict["probe_size"] = sess['probe_size']
            ids_int = np.array(ids, dtype=int)
            dir_table = sess["direction"]
            task_dict["direction"] = dir_table[ids_int - 1]
            rf_tasks.append(task_dict)
            
        if task == "plaid":
            sess = neural_data["session"][i]
            task_dict["probe_size"] = sess['probe_size']
            task_dict["task_mod"] = sess["task_mod"]
            ids_int = np.array(ids, dtype=int)
            dir_table = sess["direction"]
            plaid_table = sess["plaid"]

            task_dict["direction"] = dir_table[ids_int - 1]
            task_dict["plaid"] = plaid_table[ids_int - 1]
            task_dict["probe_class"] = "equal_contrast" if np.any(np.array(plaid_table) == 2) else "unequal_contrast"
            plaid_tasks.append(task_dict)
            
        if task == "video": 
            sess = neural_data["session"][i]
            
            task_dict["video_id"] = ids
            task_dict["probe_order"] = probe_idx_out

            is_train_trial = sess["is_train_trial"]
            task_dict["is_train_probe"] = np.array(is_train_trial, dtype=bool)[trial_idx_out]

            task_dict["video_dims_px"] = sess["video_dims_px"]
            task_dict["fix_coord_deg"] = sess["fix_coord_deg"]
            
            deg2pix = sess["deg2pix_screen"]
            dispcenter = sess["dispcenter_px"]
            vdims = sess["video_dims_px"]

            screen_width_px = (np.array(dispcenter).squeeze()[0]) * 2.0
            screenpx_to_vidpx = screen_width_px / float(vdims[0]) # note this is always 3
            assert np.abs(screenpx_to_vidpx - 3) < 0.1
            task_dict["px_per_deg"] = float(deg2pix) / float(screenpx_to_vidpx) # so we're really just dividing by 3 here!
            video_tasks.append(task_dict)
        
        # task-specific baseline duration --- first-probe onset for that task since it is aligned to fixation onset
        baseline_duration = int(1000*np.floor(10*np.nanmean(probe_onsets[:, 0]))/10) # rounded to nearest 10 ms 
        print(f"baseline duration: {baseline_duration}")

        probe_onsets = np.zeros_like(probe_onsets)
        probe_onsets[:, 1:] = np.nan
        spikes, _, _, _, _, _ = fetch_probe_spikes(spike_train, times, probe_onsets, baseline_duration, trial_correct, probe_coords, probe_ids)
        spikes = spikes[:, PROBE_TIME_BEFORE:-PROBE_TIME_AFTER, :]
        baseline_dict = {"spikes": spikes, "task_index": i}
        
        baseline_tasks.append(baseline_dict)

    # preserve neuron-related information from our trial-aligned version
    for key in neural_data['neuron'][0].keys():
        if key != 'spike_train' and key != 'times': 
            neuron_data[key] = neural_data['neuron'][0][key]
            
    neuron_data['baseline_task'] = baseline_tasks
    neuron_data["rf_task"] = rf_tasks    
    neuron_data["plaid_task"] = plaid_tasks
    neuron_data["video_task"] = video_tasks

    keys_to_remove = [
        'contam_pct', 
        'amps', 
        'mchan_xs', 
        'mchan_ys', 
        'mchan_waveforms', 
        'num_neurons', 
        'chan_best'
    ]

    for key in keys_to_remove:
        neuron_data.pop(key, None)

    with gzip.open(f"../data/output_{date}.pkl_gz", "wb") as input_file:
        pickle.dump(neuron_data, input_file, protocol=pickle.HIGHEST_PROTOCOL)
        print("done!")

020425
assumed stimulus duration: 100 ms
derived stimulus duration: 100 ms
baseline duration: 300
assumed stimulus duration: 200 ms
derived stimulus duration: 200 ms
baseline duration: 300
done!
013125
assumed stimulus duration: 100 ms
derived stimulus duration: 100 ms
baseline duration: 300
assumed stimulus duration: 200 ms
derived stimulus duration: 200 ms
baseline duration: 300
done!
012925
assumed stimulus duration: 100 ms
derived stimulus duration: 100 ms
baseline duration: 300
assumed stimulus duration: 200 ms
derived stimulus duration: 200 ms
baseline duration: 300
done!
013025
assumed stimulus duration: 100 ms
derived stimulus duration: 100 ms
baseline duration: 300
assumed stimulus duration: 200 ms
derived stimulus duration: 200 ms
baseline duration: 300
done!
021125
assumed stimulus duration: 100 ms
derived stimulus duration: 100 ms
baseline duration: 300
assumed stimulus duration: 200 ms
derived stimulus duration: 200 ms
baseline duration: 300
done!
040925
assumed stimulus d